# MIMIC-CXR Pilot Dataset 구성 및 검증

## 1. 작업 목적
전체 MIMIC-CXR-JPG 다운로드 완료 전, 소규모 모델 학습 테스트를 위해 정상 다운로드된 X-ray 이미지 1,000장을 선정하고 학습·검증·평가 데이터셋을 구성한다.

## 2. 데이터 구성
- **데이터 출처:** MIMIC-CXR-JPG (CheXpert Labels)
- **선정 기준:** 공식 Train CSV에 포함된 정상 JPG 이미지
- **Pilot 데이터:** 총 1,000장
- **분할 방식:** 환자 단위(`subject_id`) 분할, Seed=42

| 구분 | 이미지 수 | 환자 수 |
|---|---:|---:|
| Train | 800 | 728 |
| Validation | 100 | 91 |
| Test | 100 | 91 |

## 3. 주요 검증 내용
- JPG 파일 존재 여부 및 이미지 정상 디코딩 확인
- Train / Validation / Test 간 환자 중복 0명 확인
- Cardiomegaly, Edema, Pleural Effusion의 클래스 분포 확인
- Pilot CSV 저장 및 이미지 경로 연결 검증

## 4. 결과 및 활용
- **저장 경로:** `data/processed/pilot/`
- **생성 파일:** `pilot_train.csv`, `pilot_val.csv`, `pilot_test.csv`
- **후속 작업:** 기존 `CXR_Preprocessing.ipynb`에 Pilot CSV를 연결하여 DenseNet121 소규모 학습 테스트 진행

※ Pilot 데이터는 공식 Train 데이터에서 추출한 임시 실험용 데이터이며, 최종 모델 성능 평가는 별도의 공식 Validation/Test 데이터로 수행한다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

PROJECT_DIR = Path("/home/team30103/chest-xray-multilabel")
IMAGE_ROOT = Path("/home/team30103/physionet_data")

CSV_PATH = PROJECT_DIR / "data/processed/train.csv"
OUTPUT_DIR = PROJECT_DIR / "data/processed/pilot"

PILOT_SIZE = 1000
SEED = 42

LABELS = ["Cardiomegaly", "Edema", "Pleural Effusion"]

df = pd.read_csv(CSV_PATH)

print("공식 Train 이미지:", len(df))
print("CSV 파일 확인:", CSV_PATH.exists())
print("이미지 폴더 확인:", IMAGE_ROOT.exists())

공식 Train 이미지: 10000
CSV 파일 확인: True
이미지 폴더 확인: True


In [2]:
# ============================================================
# 2. 정상 다운로드된 JPG 이미지 선별
# ============================================================

valid_rows = []
corrupt_rows = []
missing_count = 0

for row in df.to_dict("records"):
    path = IMAGE_ROOT / f"{row['dicom_id']}.jpg"

    if not path.is_file():
        missing_count += 1
        continue

    try:
        # JPG 파일 구조 검사
        with Image.open(path) as img:
            img.verify()

        # 실제 이미지 디코딩 검사
        with Image.open(path) as img:
            img.load()

        valid_rows.append(row)

    except Exception as e:
        corrupt_rows.append({
            "dicom_id": row["dicom_id"],
            "error": str(e)
        })

valid_df = pd.DataFrame(valid_rows)

print("===== 이미지 검증 결과 =====")
print(f"정상 이미지: {len(valid_df):,}장")
print(f"누락 이미지: {missing_count:,}장")
print(f"손상 이미지: {len(corrupt_rows):,}장")

assert len(valid_df) >= PILOT_SIZE, "정상 이미지가 1,000장 미만입니다."

===== 이미지 검증 결과 =====
정상 이미지: 1,647장
누락 이미지: 8,348장
손상 이미지: 5장


In [3]:
# ============================================================
# 3. Pilot 1,000장 선정 및 환자 단위 분할
# ============================================================

rng = np.random.default_rng(SEED)

# 환자별로 이미지 묶기
patient_groups = [
    group.copy()
    for _, group in valid_df.groupby("subject_id")
]

rng.shuffle(patient_groups)

# 정상 이미지에서 1,000장 선정
selected = []
remaining = PILOT_SIZE

for group in patient_groups:
    if remaining == 0:
        break

    take = min(len(group), remaining)
    selected.append(group.iloc[:take])
    remaining -= take

pilot_df = pd.concat(selected, ignore_index=True)

assert len(pilot_df) == 1000
assert pilot_df["dicom_id"].is_unique

# 환자 그룹을 다시 섞어 분할
groups = [
    group.copy()
    for _, group in pilot_df.groupby("subject_id")
]
rng.shuffle(groups)

targets = {"train": 800, "val": 100, "test": 100}
splits = {"train": [], "val": [], "test": []}
counts = {"train": 0, "val": 0, "test": 0}

# 이미지 장수 차이를 줄이도록 큰 그룹부터 배정
groups.sort(key=len, reverse=True)

for group in groups:
    size = len(group)

    # 목표 장수에 대한 상대적 부족분이 큰 세트에 배정
    best = max(
        targets,
        key=lambda name: (
            (targets[name] - counts[name]) / targets[name]
        )
    )

    splits[best].append(group)
    counts[best] += size

pilot_train = pd.concat(splits["train"], ignore_index=True)
pilot_val = pd.concat(splits["val"], ignore_index=True)
pilot_test = pd.concat(splits["test"], ignore_index=True)

# 환자 중복 검증
assert set(pilot_train.subject_id).isdisjoint(pilot_val.subject_id)
assert set(pilot_train.subject_id).isdisjoint(pilot_test.subject_id)
assert set(pilot_val.subject_id).isdisjoint(pilot_test.subject_id)

assert len(pilot_train) + len(pilot_val) + len(pilot_test) == 1000

print("===== Pilot 데이터 분할 결과 =====")
for name, part in [
    ("Train", pilot_train),
    ("Validation", pilot_val),
    ("Test", pilot_test)
]:
    print(
        f"{name}: {len(part)}장 / "
        f"환자 {part.subject_id.nunique()}명"
    )

print("환자 중복: 0명")

===== Pilot 데이터 분할 결과 =====
Train: 800장 / 환자 728명
Validation: 100장 / 환자 91명
Test: 100장 / 환자 91명
환자 중복: 0명


In [4]:
# ============================================================
# 4. Pilot 데이터 클래스 분포 확인
# ============================================================

datasets = {
    "Train": pilot_train,
    "Validation": pilot_val,
    "Test": pilot_test
}

for label in LABELS:
    print(f"\n========== {label} ==========")

    rows = []

    for name, data in datasets.items():
        counts = data[label].value_counts()

        rows.append({
            "Split": name,
            "Negative (0)": counts.get(0, 0),
            "Positive (1)": counts.get(1, 0),
            "Uncertain (-1)": counts.get(-1, 0)
        })

    display(pd.DataFrame(rows))


========== Cardiomegaly ==========


,Split,Negative (0),Positive (1),Uncertain (-1)
0,Train,454,300,46
1,Validation,56,36,8
2,Test,51,44,5



========== Edema ==========


,Split,Negative (0),Positive (1),Uncertain (-1)
0,Train,483,211,106
1,Validation,66,25,9
2,Test,60,27,13



========== Pleural Effusion ==========


,Split,Negative (0),Positive (1),Uncertain (-1)
0,Train,383,377,40
1,Validation,46,49,5
2,Test,54,41,5


In [5]:
# ============================================================
# 5. Pilot CSV 저장
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pilot_train.to_csv(
    OUTPUT_DIR / "pilot_train.csv",
    index=False
)

pilot_val.to_csv(
    OUTPUT_DIR / "pilot_val.csv",
    index=False
)

pilot_test.to_csv(
    OUTPUT_DIR / "pilot_test.csv",
    index=False
)

print("===== Pilot CSV 저장 완료 =====")

for name in ["pilot_train.csv", "pilot_val.csv", "pilot_test.csv"]:
    path = OUTPUT_DIR / name
    data = pd.read_csv(path)

    print(f"{name}: {len(data)}장 | 저장 확인: {path.exists()}")

print("\n저장 경로:", OUTPUT_DIR)

===== Pilot CSV 저장 완료 =====
pilot_train.csv: 800장 | 저장 확인: True
pilot_val.csv: 100장 | 저장 확인: True
pilot_test.csv: 100장 | 저장 확인: True

저장 경로: /home/team30103/chest-xray-multilabel/data/processed/pilot


In [6]:
# ============================================================
# 6. Pilot Train / Val / Test 이미지 경로 확인
# ============================================================

pilot_datasets = {
    "Train": pd.read_csv(OUTPUT_DIR / "pilot_train.csv"),
    "Validation": pd.read_csv(OUTPUT_DIR / "pilot_val.csv"),
    "Test": pd.read_csv(OUTPUT_DIR / "pilot_test.csv"),
}

for name, data in pilot_datasets.items():
    image_paths = [
        IMAGE_ROOT / f"{dicom_id}.jpg"
        for dicom_id in data["dicom_id"]
    ]

    missing = [p for p in image_paths if not p.is_file()]

    print(f"{name}: {len(data)}장")
    print(f"  이미지 존재: {len(data) - len(missing)}장")
    print(f"  이미지 누락: {len(missing)}장")

    assert len(missing) == 0, f"{name} 이미지 누락 발생!"

print("\nPilot 이미지 경로 확인 완료!")

Train: 800장
  이미지 존재: 800장
  이미지 누락: 0장
Validation: 100장
  이미지 존재: 100장
  이미지 누락: 0장
Test: 100장
  이미지 존재: 100장
  이미지 누락: 0장

Pilot 이미지 경로 확인 완료!


## Pilot 데이터 준비 결과

- 정상 MIMIC-CXR-JPG 1,000장 선정
- Train 800장 / Validation 100장 / Test 100장
- 환자 중복 0명
- Cardiomegaly, Edema, Pleural Effusion 클래스 분포 확인
- Pilot CSV 3개 저장 및 실제 이미지 경로 검증 완료

### 저장 파일
- data/processed/pilot/pilot_train.csv
- data/processed/pilot/pilot_val.csv
- data/processed/pilot/pilot_test.csv

### 후속 작업
기존 CXR_Preprocessing.ipynb의 전처리 파이프라인을
Pilot CSV에 연결하여 DenseNet121 소규모 학습 진행

※ Pilot 데이터는 공식 Train에서 추출한 임시 실험용 데이터임.